# Resume & Job Data Exploration
## Phase 2: Data Exploration & Understanding

This notebook explores our generated resume and job description datasets to understand:
- Data quality and structure
- Feature distributions
- Potential matching patterns

In [ ]:
# Import required libraries for data analysis and visualization
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import numpy as np

# Set visualization style
plt.style.use('default')
sns.set_palette("husl")

---
# 📊 PHASE 2: DATA EXPLORATION & UNDERSTANDING
---

**Goal**: Understand our generated resume and job description datasets
- Data quality and structure
- Feature distributions  
- Potential matching patterns

## 2.1 Load and Inspect Data

In [ ]:
# Load resume data from JSON file
with open('data/resumes.json', 'r') as f:
    resumes_data = json.load(f)

# Load job descriptions from JSON file  
with open('data/jobs.json', 'r') as f:
    jobs_data = json.load(f)

print(f"Loaded {len(resumes_data)} resumes and {len(jobs_data)} job descriptions")

In [ ]:
# Convert JSON data to pandas DataFrames for easier analysis
resumes_df = pd.DataFrame(resumes_data)
jobs_df = pd.DataFrame(jobs_data)

print("Resume DataFrame Info:")
print(resumes_df.info())
print("\nFirst 3 resumes:")
print(resumes_df.head(3))

In [ ]:
print("Job DataFrame Info:")
print(jobs_df.info())
print("\nFirst 3 jobs:")
print(jobs_df.head(3))

## 2.2 Basic Statistics

In [ ]:
# Analyze resume experience distribution
print("=== RESUME STATISTICS ===")
print(f"Total resumes: {len(resumes_df)}")
print(f"Unique names: {resumes_df['name'].nunique()}")
print(f"Experience years - Min: {resumes_df['experience_years'].min()}, Max: {resumes_df['experience_years'].max()}")
print(f"Average experience: {resumes_df['experience_years'].mean():.1f} years")

# Count role categories in resumes
print("\nRole distribution:")
print(resumes_df['role_category'].value_counts())

In [ ]:
# Analyze job posting statistics
print("=== JOB STATISTICS ===")
print(f"Total jobs: {len(jobs_df)}")
print(f"Unique companies: {jobs_df['company'].nunique()}")
print(f"Experience range - Min: {jobs_df['min_experience'].min()}-{jobs_df['max_experience'].min()}, Max: {jobs_df['min_experience'].max()}-{jobs_df['max_experience'].max()}")

# Count job titles
print("\nJob title distribution:")
print(jobs_df['title'].value_counts())

## 2.3 Skills Analysis

In [ ]:
# Extract and count all skills from resumes
# Flatten the list of skills from all resumes
all_resume_skills = []
for skills_list in resumes_df['skills']:
    all_resume_skills.extend(skills_list)

# Count frequency of each skill in resumes
resume_skill_counts = Counter(all_resume_skills)
print("Top 10 skills in resumes:")
for skill, count in resume_skill_counts.most_common(10):
    print(f"{skill}: {count}")

In [ ]:
# Extract and count all skills from job descriptions
all_job_skills = []
for skills_list in jobs_df['required_skills']:
    all_job_skills.extend(skills_list)

# Count frequency of each skill in job postings
job_skill_counts = Counter(all_job_skills)
print("Top 10 skills in job descriptions:")
for skill, count in job_skill_counts.most_common(10):
    print(f"{skill}: {count}")

## 2.4 Data Visualizations

In [ ]:
# Create subplots for experience distribution comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Plot resume experience distribution
ax1.hist(resumes_df['experience_years'], bins=10, alpha=0.7, color='skyblue', edgecolor='black')
ax1.set_title('Resume Experience Distribution')
ax1.set_xlabel('Years of Experience')
ax1.set_ylabel('Count')
ax1.grid(True, alpha=0.3)

# Plot job experience requirements (using min_experience as proxy)
ax2.hist(jobs_df['min_experience'], bins=10, alpha=0.7, color='lightcoral', edgecolor='black')
ax2.set_title('Job Minimum Experience Requirements')
ax2.set_xlabel('Minimum Years Required')
ax2.set_ylabel('Count')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Create skill comparison visualization
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

# Top resume skills bar chart
top_resume_skills = dict(resume_skill_counts.most_common(8))
ax1.bar(top_resume_skills.keys(), top_resume_skills.values(), color='lightblue', alpha=0.8)
ax1.set_title('Most Common Skills in Resumes')
ax1.set_ylabel('Frequency')
ax1.tick_params(axis='x', rotation=45)

# Top job skills bar chart
top_job_skills = dict(job_skill_counts.most_common(8))
ax2.bar(top_job_skills.keys(), top_job_skills.values(), color='lightcoral', alpha=0.8)
ax2.set_title('Most Requested Skills in Job Descriptions')
ax2.set_ylabel('Frequency')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 2.5 Data Quality Assessment

In [ ]:
# Check for missing values and data quality issues
print("=== DATA QUALITY CHECK ===")
print("\nResume data missing values:")
print(resumes_df.isnull().sum())

print("\nJob data missing values:")
print(jobs_df.isnull().sum())

# Check for empty skill lists (potential data quality issue)
empty_resume_skills = sum(1 for skills in resumes_df['skills'] if len(skills) == 0)
empty_job_skills = sum(1 for skills in jobs_df['required_skills'] if len(skills) == 0)

print(f"\nResumes with no skills: {empty_resume_skills}")
print(f"Jobs with no required skills: {empty_job_skills}")

In [ ]:
# Calculate average number of skills per resume/job
resume_skill_lengths = [len(skills) for skills in resumes_df['skills']]
job_skill_lengths = [len(skills) for skills in jobs_df['required_skills']]

print(f"Average skills per resume: {np.mean(resume_skill_lengths):.1f}")
print(f"Average required skills per job: {np.mean(job_skill_lengths):.1f}")

print(f"\nSkill count ranges:")
print(f"Resumes: {min(resume_skill_lengths)}-{max(resume_skill_lengths)} skills")
print(f"Jobs: {min(job_skill_lengths)}-{max(job_skill_lengths)} skills")

## 2.6 Initial Matching Analysis

In [ ]:
# Simple skill overlap analysis for potential matching
def calculate_skill_overlap(resume_skills, job_skills):
    """Calculate percentage of job skills that candidate has"""
    resume_set = set(resume_skills)
    job_set = set(job_skills)
    overlap = len(resume_set.intersection(job_set))
    return overlap / len(job_set) if job_set else 0

# Test with first few resumes and jobs
print("=== SAMPLE MATCHING ANALYSIS ===")
for i in range(min(3, len(resumes_df))):
    resume = resumes_df.iloc[i]
    print(f"\nResume {i+1}: {resume['name']} ({resume['role_category']}, {resume['experience_years']} years)")
    print(f"Skills: {resume['skills']}")
    
    for j in range(min(2, len(jobs_df))):
        job = jobs_df.iloc[j]
        overlap = calculate_skill_overlap(resume['skills'], job['required_skills'])
        print(f"  Match with {job['title']}: {overlap:.1%} skill overlap")

## Summary & Next Steps

Based on this exploration:
1. **Data Quality**: Check if data looks clean and complete
2. **Feature Distribution**: Understand skill and experience patterns
3. **Matching Potential**: See initial overlap between resumes and jobs

**Next Phase**: Move to data preprocessing and feature engineering